In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=2, sci_mode=False, linewidth=120)

print("torch", torch.__version__)

torch 2.14.1+cpu


In [1]:
def show(mat, title="", row_labels=None, col_labels=None, decimals=2):
    """Print a 2-D tensor with token names on the rows and columns."""
    row_labels = row_labels or TOKENS
    col_labels = col_labels or TOKENS
    if title:
        print(title)
    print("       " + "".join(f"{c:>8}" for c in col_labels))
    for name, row in zip(row_labels, mat.tolist()):
        body = "".join(f"{val:8.{decimals}f}" for val in row)
        print(f"{name:>6} {body}")
    print()

In [4]:
TOKENS  = ["the", "cat", "sat", "on", "mat"]

B       = 1
T       = len(TOKENS)
d_model = 8
n_heads = 2
d_k     = d_model // n_heads

x = torch.randn(B, T, d_model)

print("tokens :", TOKENS)
print("x.shape:", tuple(x.shape), " (batch, time, width)")
print()
print("x[0] -- one row per token:")
print(x)

tokens : ['the', 'cat', 'sat', 'on', 'mat']
x.shape: (1, 5, 8)  (batch, time, width)

x[0] -- one row per token:
tensor([[[ 0.21, -0.45, -0.57, -0.56,  0.59,  1.54,  0.51, -0.59],
         [-1.33,  0.19, -0.07, -0.49, -1.50, -0.19,  0.45,  1.33],
         [ 1.51,  2.08,  1.71,  2.38, -1.13, -0.32, -1.09, -0.09],
         [ 1.65, -1.36,  0.34,  0.52, -2.61, -1.70, -0.23,  0.28],
         [ 0.25,  0.08,  0.34,  0.45,  0.46, -0.87,  0.78, -0.93]]])


In [6]:
col_labels = [ f"d{i}" for i in range(5) ]
show(x[0][:, :5], "x, first 5 columns", col_labels=col_labels)

x, first 5 columns
             d0      d1      d2      d3      d4
   the     0.21   -0.45   -0.57   -0.56    0.59
   cat    -1.33    0.19   -0.07   -0.49   -1.50
   sat     1.51    2.08    1.71    2.38   -1.13
    on     1.65   -1.36    0.34    0.52   -2.61
   mat     0.25    0.08    0.34    0.45    0.46



In [7]:
naive_scores = x[0] @ x[0].T

print("x[0].shape:", tuple(x[0].shape))
print("naive_scores.shape:", tuple(naive_scores.shape))

show(naive_scores, "naive_scores[i][j] = how much token i matches token j")

x[0].shape: (5, 8)
naive_scores.shape: (5, 5)
naive_scores[i][j] = how much token i matches token j
            the     cat     sat      on     mat
   the     4.22   -1.79   -4.59   -3.99   -0.55
   cat    -1.79    6.27   -1.76    1.79   -1.96
   sat    -4.59   -1.76   17.76    5.18    1.18
    on    -3.99    1.79    5.18   14.79    0.49
   mat    -0.55   -1.96    1.18    0.49    2.81



In [8]:

W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)
W_o = nn.Linear(d_model, d_model)

q = W_q(x)
k = W_k(x)
v = W_v(x)

mask = torch.tril(
    torch.ones(T, T, dtype=torch.bool)
)

print("x.shape:", tuple(x.shape))
print("q.shape:", tuple(q.shape))
print("k.shape:", tuple(k.shape))
print("v.shape:", tuple(v.shape))
print()

x.shape: (1, 5, 8)
q.shape: (1, 5, 8)
k.shape: (1, 5, 8)
v.shape: (1, 5, 8)



In [9]:
print(x[0][2])
print(q[0][2])
print(k[0][2])

tensor([ 1.51,  2.08,  1.71,  2.38, -1.13, -0.32, -1.09, -0.09])
tensor([-0.62,  0.41, -0.30,  0.16, -1.95, -0.95,  0.55, -1.33], grad_fn=<SelectBackward0>)
tensor([ 0.37,  2.09, -0.20,  0.49, -0.81, -1.48,  0.08,  0.92], grad_fn=<SelectBackward0>)


In [10]:

scores = q @ k.transpose(-2, -1)

print("scores.shape:", tuple(scores.shape))
print()
show(scores[0], "scores = q @ k^T")

scores.shape: (1, 5, 5)

scores = q @ k^T
            the     cat     sat      on     mat
   the     0.58    0.38   -1.32   -0.74   -0.47
   cat     1.34   -2.57   -0.40   -2.56   -0.36
   sat    -1.80    1.26    2.57    1.80    1.21
    on    -0.55   -0.70    1.02    1.14    0.33
   mat    -0.78    1.97   -0.30    1.06   -0.02



In [11]:

def row(t, style="{:6.2f}"):
    return "[" + "  ".join(style.format(v) for v in t.tolist()) + "]"

small = torch.tensor([0.5, 1.2, 0.8])
big = small * 16

print("scores", row(small), "-> softmax", row(F.softmax(small, dim=-1), "{:6.1%}"))
print("scores", row(big),   "-> softmax", row(F.softmax(big,   dim=-1), "{:6.1%}"))

scores [  0.50    1.20    0.80] -> softmax [ 22.9%   46.1%   30.9%]
scores [  8.00   19.20   12.80] -> softmax [  0.0%   99.8%    0.2%]
